# **Modelado Con scikit-learn**

En esta sección se implementa el modelo de clasificación con scikit-learn utilizando un `RandomForestClassifier`.

Siguiendo la guía del proyecto, se aplicará una búsqueda de hiperparámetros mediante `GridSearchCV`, y posteriormente se evaluará el desempeño del mejor modelo utilizando diversas métricas de clasificación.

In [1]:
import time
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

## **Carga del dataset y selección de variables**

Se reconstruye el subconjunto de variables seleccionado en la etapa de preprocesamiento, con el fin de mantener este notebook autocontenido y reproducible.

In [2]:
DATA_PATH = Path(r"C:\Users\Daniel Rangel\Documents\MachineLearning\Data\accepted_2007_to_2018Q4.csv.gz")

df = pd.read_csv(DATA_PATH, low_memory=False)
df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)

num_vars = [
    "loan_amnt",
    "int_rate",
    "fico_range_high",
    "annual_inc",
    "dti",
    "revol_util",
    "open_acc",
    "total_acc"
]

cat_vars = [
    "emp_length",
    "purpose",
    "home_ownership",
    "addr_state"
]

selected_vars = num_vars + cat_vars

df_model = df[selected_vars + ["default"]].copy()

print("Dimensión del subconjunto:", df_model.shape)

Dimensión del subconjunto: (2260701, 13)


## **Partición en entrenamiento y prueba**

Se divide el dataset en entrenamiento y prueba con una proporción 80/20, utilizando estratificación sobre la variable objetivo para conservar la distribución de clases en ambos subconjuntos.

In [3]:
X = df_model[selected_vars]
y = df_model["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Distribución y_train:")
print(y_train.value_counts(normalize=True).round(4))
print("Distribución y_test:")
print(y_test.value_counts(normalize=True).round(4))

X_train: (1808560, 12)
X_test: (452141, 12)
Distribución y_train:
0    0.8812
1    0.1188
Name: default, dtype: float64
Distribución y_test:
0    0.8812
1    0.1188
Name: default, dtype: float64


## **Pipeline de preprocesamiento**

Las variables numéricas se imputan con la mediana y se escalan con `StandardScaler`, mientras que las variables categóricas se imputan con la categoría más frecuente y se codifican mediante `OneHotEncoder`.

In [4]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_vars),
        ("cat", categorical_transformer, cat_vars)
    ]
)

## **Definición del modelo y búsqueda de hiperparámetros**

Se construye un pipeline que integra el preprocesamiento y el clasificador `RandomForestClassifier`. Posteriormente, se realiza una búsqueda de hiperparámetros con `GridSearchCV`, utilizando la grilla establecida en la guía del proyecto.

In [7]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=1))
])

param_grid = {
    "classifier__n_estimators": [10, 50, 100],
    "classifier__max_depth": [5, 10, 15]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=1,
    verbose=2
)

## **Entrenamiento del modelo**

Se entrena el modelo de Random Forest utilizando `GridSearchCV`, con el fin de identificar la mejor combinación de hiperparámetros dentro de la grilla definida.

In [8]:
start_train = time.time()

grid_search.fit(X_train, y_train)

train_time = time.time() - start_train

print("Mejores parámetros:", grid_search.best_params_)
print(f"Tiempo de entrenamiento: {train_time:.2f} segundos")

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END classifier__max_depth=5, classifier__n_estimators=10; total time=  16.1s
[CV] END classifier__max_depth=5, classifier__n_estimators=10; total time=  15.4s
[CV] END classifier__max_depth=5, classifier__n_estimators=10; total time=  14.9s
[CV] END classifier__max_depth=5, classifier__n_estimators=50; total time= 1.0min
[CV] END classifier__max_depth=5, classifier__n_estimators=50; total time= 1.0min
[CV] END classifier__max_depth=5, classifier__n_estimators=50; total time=  59.5s
[CV] END classifier__max_depth=5, classifier__n_estimators=100; total time= 1.9min
[CV] END classifier__max_depth=5, classifier__n_estimators=100; total time= 1.9min
[CV] END classifier__max_depth=5, classifier__n_estimators=100; total time= 1.9min
[CV] END classifier__max_depth=10, classifier__n_estimators=10; total time=  46.2s
[CV] END classifier__max_depth=10, classifier__n_estimators=10; total time=  44.2s
[CV] END classifier__max_depth=10

## **Resultado de la búsqueda de hiperparámetros**

La búsqueda de hiperparámetros con `GridSearchCV` se completó satisfactoriamente, evaluando un total de 27 ajustes correspondientes a 9 combinaciones de parámetros con validación cruzada de 3 folds.

La mejor configuración encontrada fue:

- `max_depth = 15`
- `n_estimators = 10`

El tiempo total de entrenamiento fue de aproximadamente **8845.24 segundos**, equivalente a cerca de **2.46 horas**.

Este resultado evidencia que, aunque el flujo con scikit-learn es funcional sobre el dataset completo, el costo computacional en tiempo es elevado en un entorno local, especialmente al combinar validación cruzada, codificación de variables categóricas y búsqueda de hiperparámetros.

In [9]:
best_model = grid_search.best_estimator_
best_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['loan_amnt', 'int_rate',
                                                   'fico_range_high',
                                                   'annual_inc', 'dti',
                                                   'revol_util', 'open_acc',
                                                   'total_acc']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['emp_length', 'purpose',
                                                   'home_ownership',
                                                   'addr_state'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=15, n_estimators=10, n_jobs=1,
                                        random_state=42))])

## **Predicción sobre el conjunto de prueba**

Una vez identificado el mejor modelo mediante `GridSearchCV`, se realizan predicciones sobre el conjunto de prueba para evaluar su desempeño y medir el tiempo requerido en la etapa de inferencia.

In [10]:
start_pred = time.time()

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

pred_time = time.time() - start_pred

print(f"Tiempo de predicción: {pred_time:.2f} segundos")
print("Número de predicciones:", len(y_pred))

Tiempo de predicción: 2.68 segundos
Número de predicciones: 452141


El mejor modelo seleccionado mediante `GridSearchCV` realizó predicciones sobre el conjunto de prueba en aproximadamente **2.68 segundos**.

Este tiempo de inferencia es considerablemente menor que el tiempo de entrenamiento, lo cual es esperable en modelos de Random Forest una vez que el proceso de ajuste y selección de hiperparámetros ha concluido.

In [11]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

metrics_df = pd.DataFrame({
    "Métrica": ["Accuracy", "Precision", "Recall", "F1-score", "ROC AUC"],
    "Valor": [accuracy, precision, recall, f1, roc_auc]
})

metrics_df

c:\Users\Daniel Rangel\miniconda3\envs\ml_rangel\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


,Métrica,Valor
0,Accuracy,0.881205
1,Precision,0.000000
2,Recall,0.000000
3,F1-score,0.000000
4,ROC AUC,0.695485


## **Evaluación del modelo en el conjunto de prueba**

Los resultados muestran un comportamiento fuertemente afectado por el desbalance de clases presente en la variable objetivo.

El modelo alcanza una **accuracy de 0.8812**, valor que a primera vista parece alto. Sin embargo, esta métrica resulta engañosa en este contexto, ya que coincide prácticamente con la proporción de la clase mayoritaria en el dataset.

En contraste, las métricas de la clase positiva muestran un desempeño nulo:

- **Precision = 0.0**
- **Recall = 0.0**
- **F1-score = 0.0**

Esto indica que el modelo no predijo ningún caso como `default = 1`, clasificando todas las observaciones como pertenecientes a la clase mayoritaria (`default = 0`).

A pesar de ello, el valor de **ROC AUC = 0.6955** sugiere que el modelo sí logra cierto nivel de discriminación en las probabilidades estimadas, aunque dicha capacidad no se traduce en predicciones positivas bajo el umbral de decisión por defecto.

En consecuencia, estos resultados confirman que el desbalance de clases representa un problema importante y que será necesario considerar ajustes posteriores, como el uso de pesos de clase, modificación del umbral de decisión o estrategias de balanceo.

In [12]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Real 0", "Real 1"],
    columns=["Predicho 0", "Predicho 1"]
)

cm_df

,Predicho 0,Predicho 1
Real 0,398429,0
Real 1,53712,0


### **Interpretación de la matriz de confusión**

La matriz de confusión muestra que el modelo clasificó **todas** las observaciones del conjunto de prueba como pertenecientes a la clase `0`.

En particular:

- Los **398,429** casos reales de la clase `0` fueron clasificados correctamente.
- Los **53,712** casos reales de la clase `1` fueron clasificados incorrectamente como `0`.
- No se registró ninguna predicción de la clase positiva.

Este comportamiento explica por qué la accuracy resulta alta, pero las métricas de la clase positiva (`precision`, `recall` y `F1-score`) toman valor cero. En la práctica, esto significa que el modelo no está cumpliendo adecuadamente el objetivo del problema, ya que no logra identificar préstamos en default.

Por tanto, aunque el modelo presenta capacidad de discriminación en términos de ROC AUC, el umbral de decisión por defecto y el desbalance de clases impiden que esa información se traduzca en predicciones útiles para la clase minoritaria.

In [13]:
print(classification_report(y_test, y_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.88      1.00      0.94    398429
           1       0.00      0.00      0.00     53712

    accuracy                           0.88    452141
   macro avg       0.44      0.50      0.47    452141
weighted avg       0.78      0.88      0.83    452141



### **Interpretación del classification report**

El reporte de clasificación confirma que el modelo presenta un desempeño aceptable únicamente sobre la clase mayoritaria (`default = 0`), pero falla por completo en la identificación de la clase minoritaria (`default = 1`).

Para la clase `0`, el modelo alcanza:

- **precision = 0.88**
- **recall = 1.00**
- **f1-score = 0.94**

Esto indica que prácticamente todos los casos fueron clasificados como no default, y que la clase mayoritaria fue capturada por completo.

Sin embargo, para la clase `1`, todas las métricas son iguales a cero:

- **precision = 0.00**
- **recall = 0.00**
- **f1-score = 0.00**

Esto significa que el modelo no logró identificar ningún préstamo en default dentro del conjunto de prueba.

Asimismo, la diferencia entre el **macro average** y el **weighted average** refleja el impacto del desbalance de clases. Mientras el promedio ponderado permanece relativamente alto debido al predominio de la clase 0, el promedio macro evidencia un desempeño muy deficiente cuando ambas clases se consideran con igual importancia.

En consecuencia, aunque el flujo de modelado se ejecutó correctamente, el modelo obtenido en esta configuración no resulta adecuado para el objetivo del problema, ya que no logra detectar la clase positiva de interés.

## **Conclusión de la evaluación del modelo scikit-learn**

El modelo Random Forest implementado con scikit-learn logró entrenarse correctamente y alcanzar una accuracy alta; sin embargo, dicha métrica resulta engañosa en el contexto de un problema desbalanceado.

Las métricas de la clase positiva, la matriz de confusión y el reporte de clasificación muestran que el modelo no identifica casos de default bajo la configuración actual, clasificando todas las observaciones como pertenecientes a la clase mayoritaria.

Por tanto, los resultados sugieren que será necesario considerar estrategias adicionales para abordar el desbalance de clases, tales como el uso de pesos de clase, ajuste del umbral de decisión o técnicas de balanceo, antes de considerar este modelo como una solución adecuada al problema planteado.